In [1]:
import scanpy as sc
import omicverse as ov
import pandas as pd
ov.plot_set()


   ____            _     _    __                  
  / __ \____ ___  (_)___| |  / /__  _____________ 
 / / / / __ `__ \/ / ___/ | / / _ \/ ___/ ___/ _ \ 
/ /_/ / / / / / / / /__ | |/ /  __/ /  (__  )  __/ 
\____/_/ /_/ /_/_/\___/ |___/\___/_/  /____/\___/                                              

Version: 1.6.11, Tutorials: https://omicverse.readthedocs.io/
Dependency error: The 'phate>=1.0' distribution was not found and is required by the application


In [2]:
adata = sc.read("/home/lugli/spuccio/Projects/SP039/GBmap/Chen2021_Part2.h5ad")

In [3]:
adata = adata[adata.obs['donor_id'].isin(["PJ053", "PW016-703", "PW017-703", "PW032-710"])]

In [4]:
df_obs = pd.DataFrame(adata.obs)

In [5]:
del adata.obs

In [6]:
adata = adata.raw.to_adata()

In [7]:
adata

AnnData object with n_obs × n_vars = 4804 × 20134
    var: 'mt', 'n_cells', 'percent_cells', 'robust', 'means', 'variances', 'residual_variances', 'highly_variable_rank', 'highly_variable_features'
    uns: 'X_approximate_distribution', 'annotation_level_1_colors', 'annotation_level_2_colors', 'annotation_level_3_colors', 'batch_condition', 'default_embedding', 'donor_id_colors', 'hvg', 'leiden', 'leiden_colors', 'log1p', 'neighbors', 'pca', 'rank_genes_groups', 'scaled|original|cum_sum_eigenvalues', 'scaled|original|pca_var_ratios', 'schema_version', 'scsa_celltype_cellmarker_colors', 'scsa_celltype_panglaodb_colors', 'title', 'umap'
    obsm: 'X_harmony', 'X_pca', 'X_umap', 'scaled|original|X_pca'
    obsp: 'connectivities', 'distances'

In [8]:
#adata = adata.raw.to_adata()

In [9]:
X_counts_recovered, size_factors_sub=ov.pp.recover_counts(adata.X, 50*1e4, 50*1e5, log_base=None, 
                                                          chunk_size=10000)


100%|██████████| 4804/4804 [00:06<00:00, 696.39it/s]


In [10]:
adata.X = X_counts_recovered

In [11]:
annot = sc.queries.biomart_annotations(
    "hsapiens",
    ["external_gene_name","ensembl_gene_id", "start_position", "end_position", "chromosome_name",],
).set_index("external_gene_name")

In [12]:
annot

,ensembl_gene_id,start_position,end_position,chromosome_name
external_gene_name,,,,
MT-TF,ENSG00000210049,577,647,MT
MT-RNR1,ENSG00000211459,648,1601,MT
MT-TV,ENSG00000210077,1602,1670,MT
MT-RNR2,ENSG00000210082,1671,3229,MT
MT-TL1,ENSG00000209082,3230,3304,MT
...,...,...,...,...
SCMH1-DT,ENSG00000235358,41241772,41338644,1
LINC01740,ENSG00000228067,212467563,212556085,1
SLC44A3-AS1,ENSG00000293271,94585556,94855426,1


In [13]:
adata.var.columns

Index(['mt', 'n_cells', 'percent_cells', 'robust', 'means', 'variances',
       'residual_variances', 'highly_variable_rank',
       'highly_variable_features'],
      dtype='object')

In [14]:
adata.var = adata.var[['mt', 'n_cells', 'percent_cells', 'robust', 'means', 'variances',
       'residual_variances']]

In [15]:
adata.var 

,mt,n_cells,percent_cells,robust,means,variances,residual_variances
feature_name,,,,,,,
ZNF470-DT,False,72,0.756461,True,0.009158,0.012045,1.370489
ZNF367,False,413,4.339147,True,0.050126,0.061261,1.213519
SULT1B1,False,7,0.073545,True,0.001277,0.002413,2.287334
PABPN1P1,False,223,2.342929,True,0.027370,0.033951,1.248019
HDHD2,False,949,9.970582,True,0.117431,0.136626,1.188500
...,...,...,...,...,...,...,...
LINC01886,False,14,0.147090,True,0.001420,0.001429,0.916108
LINC01732,False,6,0.063038,True,0.000767,0.001003,1.353200
LINC01775,False,11,0.115570,True,0.001036,0.000971,0.814317


In [16]:
df_tmp = pd.merge(adata.var , annot, left_index=True, right_index=True, how='left')

In [17]:
df_tmp = df_tmp.reset_index().drop_duplicates(['feature_name']).set_index(['feature_name'])

In [18]:
adata.var = df_tmp

In [19]:
adata = adata[:,adata.var['chromosome_name'].isin(["1","2","3","4","5","6","7","8","9","10","11","12","13","14","15","16","17","18","19","20","21","22","X","Y","MT"])]

In [20]:
adata

View of AnnData object with n_obs × n_vars = 4804 × 16502
    var: 'mt', 'n_cells', 'percent_cells', 'robust', 'means', 'variances', 'residual_variances', 'ensembl_gene_id', 'start_position', 'end_position', 'chromosome_name'
    uns: 'X_approximate_distribution', 'annotation_level_1_colors', 'annotation_level_2_colors', 'annotation_level_3_colors', 'batch_condition', 'default_embedding', 'donor_id_colors', 'hvg', 'leiden', 'leiden_colors', 'log1p', 'neighbors', 'pca', 'rank_genes_groups', 'scaled|original|cum_sum_eigenvalues', 'scaled|original|pca_var_ratios', 'schema_version', 'scsa_celltype_cellmarker_colors', 'scsa_celltype_panglaodb_colors', 'title', 'umap'
    obsm: 'X_harmony', 'X_pca', 'X_umap', 'scaled|original|X_pca'
    obsp: 'connectivities', 'distances'

In [21]:
adata.obs['donor_id'] = df_obs['donor_id']

In [22]:
metadata_data = {
    'Author': ['Chen2021'] * 4,
    'donor_id': ["PJ053", "PW016-703", "PW017-703", "PW032-710"],
    'stage': ['Primary'] * 4,
    'assay': ['microwell-seq'] * 4,
    'tissue': ['brain', 'left frontal lobe', 'temporoparietal junction', 'left frontal lobe'],
    'Cells': ['Total'] * 4,
    'Method': ['cell'] * 4
}

metadata_df = pd.DataFrame(metadata_data)

# Display the metadata DataFrame
print(metadata_df)

     Author   donor_id    stage          assay                    tissue  \
0  Chen2021      PJ053  Primary  microwell-seq                     brain   
1  Chen2021  PW016-703  Primary  microwell-seq         left frontal lobe   
2  Chen2021  PW017-703  Primary  microwell-seq  temporoparietal junction   
3  Chen2021  PW032-710  Primary  microwell-seq         left frontal lobe   

   Cells Method  
0  Total   cell  
1  Total   cell  
2  Total   cell  
3  Total   cell  


In [23]:
merged_obs_df = pd.merge(pd.DataFrame(adata.obs), metadata_df, left_on='donor_id', right_on='donor_id', how='left')

# Display the merged dataframe
print(merged_obs_df)

       donor_id    Author    stage          assay             tissue  Cells  \
0         PJ053  Chen2021  Primary  microwell-seq              brain  Total   
1         PJ053  Chen2021  Primary  microwell-seq              brain  Total   
2         PJ053  Chen2021  Primary  microwell-seq              brain  Total   
3         PJ053  Chen2021  Primary  microwell-seq              brain  Total   
4         PJ053  Chen2021  Primary  microwell-seq              brain  Total   
...         ...       ...      ...            ...                ...    ...   
4799  PW032-710  Chen2021  Primary  microwell-seq  left frontal lobe  Total   
4800  PW032-710  Chen2021  Primary  microwell-seq  left frontal lobe  Total   
4801  PW032-710  Chen2021  Primary  microwell-seq  left frontal lobe  Total   
4802  PW032-710  Chen2021  Primary  microwell-seq  left frontal lobe  Total   
4803  PW032-710  Chen2021  Primary  microwell-seq  left frontal lobe  Total   

     Method  
0      cell  
1      cell  
2      ce

In [24]:
df_obs = df_obs[['donor_id','n_genes','nUMIs','annotation_level_1', 'annotation_level_2','annotation_level_3','scsa_celltype_cellmarker', 'scsa_celltype_panglaodb','cell_type']]

In [25]:
df_obs

,donor_id,n_genes,nUMIs,annotation_level_1,annotation_level_2,annotation_level_3,scsa_celltype_cellmarker,scsa_celltype_panglaodb,cell_type
PJ053_TGCATTCCGCGT-0-1,PJ053,4116,2714.198975,Neoplastic,Stem-like,NPC-like,Oligodendrocyte,Oligodendrocytes,malignant cell
PJ053_GCTGAGATACGA-0-1,PJ053,3969,2611.666992,Neoplastic,Differentiated-like,MES-like,Astrocyte,Astrocytes,malignant cell
PJ053_CACCAGCCAGGA-0-1,PJ053,3709,2662.290039,Neoplastic,Stem-like,OPC-like,Neuron,Photoreceptor Cells,malignant cell
PJ053_TCCTTCAGAGAT-0-1,PJ053,3363,2544.714111,Neoplastic,Differentiated-like,MES-like,Oligodendrocyte,Oligodendrocytes,malignant cell
PJ053_GCACCGTGTTTT-0-1,PJ053,3417,2563.853027,Neoplastic,Stem-like,OPC-like,Oligodendrocyte,Oligodendrocytes,malignant cell
...,...,...,...,...,...,...,...,...,...
PW032-710_CAAATGGCCACG-0-1,PW032-710,709,1345.046753,Neoplastic,Stem-like,OPC-like,M1 macrophage,Crypt Cells,malignant cell
PW032-710_CCTCCCCCCAGG-0-1,PW032-710,569,1122.364014,Non-neoplastic,Myeloid,TAM-BDM,Microglial cell,Macrophages,macrophage
PW032-710_AAGGGTGAGACT-0-1,PW032-710,643,1251.013672,Neoplastic,Differentiated-like,MES-like,Astrocyte,Astrocytes,malignant cell
PW032-710_CATACCATTAAG-0-1,PW032-710,585,1169.987793,Neoplastic,Differentiated-like,MES-like,M1 macrophage,Crypt Cells,malignant cell


In [26]:
merged_obs_df.index= df_obs.index

In [27]:
merged_obs_df.columns

Index(['donor_id', 'Author', 'stage', 'assay', 'tissue', 'Cells', 'Method'], dtype='object')

In [28]:
merged_obs_df = pd.merge(merged_obs_df, df_obs,right_index=True,left_index=True, how='left')

In [29]:
merged_obs_df.columns

Index(['donor_id_x', 'Author', 'stage', 'assay', 'tissue', 'Cells', 'Method',
       'donor_id_y', 'n_genes', 'nUMIs', 'annotation_level_1',
       'annotation_level_2', 'annotation_level_3', 'scsa_celltype_cellmarker',
       'scsa_celltype_panglaodb', 'cell_type'],
      dtype='object')

In [30]:
del merged_obs_df['donor_id_y']

In [31]:
merged_obs_df.columns = ['donor_id', 'Author', 'stage', 'assay', 'tissue', 'Cells', 'Method',
                         'n_genes', 'nUMIs', 'annotation_level_1',
       'annotation_level_2', 'annotation_level_3', 'scsa_celltype_cellmarker',
       'scsa_celltype_panglaodb', 'cell_type']

In [32]:
merged_obs_df.columns

Index(['donor_id', 'Author', 'stage', 'assay', 'tissue', 'Cells', 'Method',
       'n_genes', 'nUMIs', 'annotation_level_1', 'annotation_level_2',
       'annotation_level_3', 'scsa_celltype_cellmarker',
       'scsa_celltype_panglaodb', 'cell_type'],
      dtype='object')

In [33]:
adata.obs = merged_obs_df

In [34]:
ov.pp.score_genes_cell_cycle(adata,species='human')

calculating cell cycle phase
computing score 'S_score'
    finished: added
    'S_score', score of gene set (adata.obs).
    769 total control genes are used. (0:00:00)
computing score 'G2M_score'
    finished: added
    'G2M_score', score of gene set (adata.obs).
    686 total control genes are used. (0:00:00)
-->     'phase', cell cycle phase (adata.obs)


In [35]:
adata.write("/home/lugli/spuccio/Projects/SP039/GBmap/Chen2021_Part3.h5ad")